# Solver Comparison (NetworkX + Pathfinding + Grid)

This notebook generates a single sample at a chosen resolution using `passage.generate.generate_sample`,
then compares NetworkX, Pathfinding and internal Grid (NumPy) solvers from `passage.generate.find_path`.

It records computation time, summarizes results, and visualizes all paths on the elevation map.

In [ ]:
import json
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

# Add src to path for imports
ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "src"))

In [ ]:
from passage.generate import generate_sample, find_path

In [ ]:
# Solvers list (moved to top for visibility)
CONFIG_PATH = ROOT / "config" / "config.yaml"
DEFAULT_COST_MODEL = "elevation"

try:
    with open(CONFIG_PATH) as f:
        _config = yaml.safe_load(f) or {}
    _solver_cfg = _config.get("solver", {})
    COST_MODEL = (
        _solver_cfg.get("grid", {}).get("cost_model")
        or _solver_cfg.get("networkx", {}).get("cost_model")
        or _solver_cfg.get("pathfinding", {}).get("cost_model")
        or DEFAULT_COST_MODEL
    )
except Exception:
    COST_MODEL = DEFAULT_COST_MODEL

SOLVERS = [
    # # NetworkX
    # {"name": "networkx:astar", "heuristic": "euclidean", "cost_model": COST_MODEL, "label": "NetworkX A* (Euclidean)"},
    # {"name": "networkx:astar_bi", "heuristic": "euclidean", "cost_model": COST_MODEL, "label": "NetworkX A* (Bidirectional)"},
    # {"name": "networkx:dijkstra", "cost_model": COST_MODEL, "label": "NetworkX Dijkstra"},
    # {"name": "networkx:dijkstra_bi", "cost_model": COST_MODEL, "label": "NetworkX Dijkstra (Bidirectional)"},
    # {"name": "networkx:bellman_ford", "cost_model": COST_MODEL, "label": "NetworkX Bellman-Ford"},
    # {"name": "networkx:unweighted", "cost_model": COST_MODEL, "label": "NetworkX Unweighted (BFS)"},
    # # Pathfinding
    # {"name": "pathfinding:astar", "heuristic": "euclidean", "cost_model": COST_MODEL, "label": "Pathfinding A* (Euclidean)"},
    # {"name": "pathfinding:dijkstra", "cost_model": COST_MODEL, "label": "Pathfinding Dijkstra"},
    # {"name": "pathfinding:best_first", "heuristic": "euclidean", "cost_model": COST_MODEL, "label": "Pathfinding Best-First"},
    # Grid (NumPy-based internal solvers)
    {"name": "grid:astar", "heuristic": "euclidean", "cost_model": COST_MODEL, "label": "Grid A* (NumPy)"},
    {"name": "grid:dijkstra", "cost_model": COST_MODEL, "label": "Grid Dijkstra (NumPy)"},
]

## Configuration

Choose a resolution and where to store the generated sample.

In [ ]:
# Sample generation settings
RESOLUTION = 4096  # 64, 128, 256, 512, 1024, 2048, 4096
SAMPLE_INDEX = 0
DISABLE_IMAGES = False
OVERRIDE_EXISTING = True

# Paths
OUTPUTS_DIR = ROOT / "outputs" / "solve"
TILES_DIR = ROOT / "outputs" / "download" / "tiles"
CALIBRATE_PATH = ROOT / "outputs" / "calibrate" / "calibrate.json"

# Solver defaults for sample generation (used only to ensure a valid path exists)
GEN_SOLVER_CONFIG = {
    "name": "grid:astar",
    "diagonal_movement": "always",
    "weight": 30.0,
    "cost_model": COST_MODEL,
    "heuristic": "euclidean",
}

# Solver comparison settings
WEIGHT = 30.0
DIAGONAL_MOVEMENT = "always"
COST_MODEL = COST_MODEL  # Pathfinding supports elevation, distance, absolute_elevation

## Generate or Load Sample

This uses `generate_sample` from `passage.generate` to create a single tensor if it does not exist yet.

In [ ]:
# Validate required inputs
if not CALIBRATE_PATH.exists():
    raise FileNotFoundError(f"Calibration file not found: {CALIBRATE_PATH}. Run 'passage calibrate' first.")
if not TILES_DIR.exists():
    raise FileNotFoundError(f"Tiles directory not found: {TILES_DIR}. Run 'passage download' first.")

# Load calibration data
with open(CALIBRATE_PATH) as f:
    calibration = json.load(f)
global_min = float(calibration["global_min"])
global_max = float(calibration["global_max"])
tile_stats_df = pd.DataFrame(calibration["tile_stats"])

# Generate sample if not present (or if OVERRIDE_EXISTING is True)
tensor_path = OUTPUTS_DIR / str(RESOLUTION) / "tensors" / f"{SAMPLE_INDEX:010d}.npy"
metadata_path = OUTPUTS_DIR / str(RESOLUTION) / "metadata" / f"{SAMPLE_INDEX:010d}.yaml"

if OVERRIDE_EXISTING or not tensor_path.exists() or not metadata_path.exists():
    OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
    args = (
        SAMPLE_INDEX,
        TILES_DIR,
        OUTPUTS_DIR,
        RESOLUTION,
        tile_stats_df,
        global_min,
        global_max,
        DISABLE_IMAGES,
        GEN_SOLVER_CONFIG,
    )
    index, success, message, duration = generate_sample(args)
    print(f"Generate sample: success={success}, message='{message}', duration={duration:.2f}s")
    if not success:
        raise RuntimeError("Sample generation failed. Try a different resolution or re-run calibration.")
else:
    print(f"Using existing tensor: {tensor_path}")

# Load tensor (shape: H, W, 2) -> channels: elevation (normalized), markers
tensor = np.load(tensor_path)
elevation_norm = tensor[:, :, 0]
markers = tensor[:, :, 1]
# elevation_raw is the unnormalized elevation in the original units
elevation_raw = elevation_norm * (global_max - global_min) + global_min

## Extract Markers

Use the marker channel to recover the start/end coordinates.

In [ ]:
marker_positions = np.argwhere(markers > 0.5)
if len(marker_positions) < 2:
    raise ValueError("Expected at least two markers in the tensor.")

start = tuple(marker_positions[0])
end = tuple(marker_positions[1])

print(f"Start: {start}")
print(f"End:   {end}")
print(f"Euclidean distance: {np.hypot(start[0] - end[0], start[1] - end[1]):.1f} pixels")

## Visualize Elevation Map

Visualize elevation map and markers.

In [ ]:
# Display elevation map with the same "terrain" colormap and show markers if available
fig, ax = plt.subplots(figsize=(8, 8))
im = ax.imshow(elevation_norm, cmap="terrain", alpha=0.95)
ax.set_title(f"Elevation (normalized) - {RESOLUTION}x{RESOLUTION}")
ax.axis("off")
cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label("Normalized elevation")

# Plot start/end markers if present (use existing `start`/`end` if defined, else derive from markers)
marker_positions = None
try:
    marker_positions = np.array([start, end])
except NameError:
    marker_positions = np.argwhere(markers > 0.5)

if marker_positions is not None and len(marker_positions) > 0:
    if marker_positions.shape[0] >= 1:
        s = tuple(marker_positions[0])
        ax.scatter([s[1]], [s[0]], c="lime", s=60, marker="o", label="Start", zorder=5)
    if marker_positions.shape[0] >= 2:
        e = tuple(marker_positions[1])
        ax.scatter([e[1]], [e[0]], c="cyan", s=60, marker="x", label="End", zorder=5)
    ax.legend(loc="upper right")

plt.tight_layout()
plt.show()

## Run Solver Comparison

Compare all NetworkX and Pathfinding solvers and record execution time.

In [ ]:
# Prebuild NetworkX graphs for each cost_model used by networkx solvers (reuse if multiple solvers use same model)
networkx_solvers = [s for s in SOLVERS if s["name"].split(":", 1)[0] == "networkx"]
G_prebuilt_map = {}
if networkx_solvers:
    from passage.generate import build_graph_from_elevation

    # Build a shared graph per cost_model (defaults to global COST_MODEL)
    cost_models = set(s.get("cost_model", COST_MODEL) for s in networkx_solvers)
    for cm in cost_models:
        G_prebuilt_map[cm] = build_graph_from_elevation(
            elevation_raw, weight=WEIGHT, diagonal_movement=DIAGONAL_MOVEMENT, cost_model=cm
        )

results = []

for solver_info in SOLVERS:
    solver_config = {
        "name": solver_info["name"],
        "diagonal_movement": DIAGONAL_MOVEMENT,
        "weight": WEIGHT,
        "cost_model": solver_info.get("cost_model", COST_MODEL),
    }
    if "heuristic" in solver_info:
        solver_config["heuristic"] = solver_info["heuristic"]

    t0 = time.perf_counter()

    backend_name = solver_info["name"].split(":", 1)[0]
    if backend_name == "networkx":
        prebuilt = G_prebuilt_map.get(solver_config["cost_model"])
        path = find_path(elevation_raw, start, end, solver_config, prebuilt_graph=prebuilt)
    else:
        path = find_path(elevation_raw, start, end, solver_config)

    elapsed = (time.perf_counter() - t0) * 1000

    results.append(
        {
            "solver": solver_info["label"],
            "backend": backend_name,
            "path_length": len(path),
            "time_ms": elapsed,
            "path": path,
        }
    )

    print(f"{solver_info['label']}: path_len={len(path)}, time={elapsed:.1f} ms")


timings_df = pd.DataFrame(
    [
        {"Solver": r["solver"], "Backend": r["backend"], "Path Length": r["path_length"], "Time (ms)": r["time_ms"]}
        for r in results
    ]
)
csv_path = OUTPUTS_DIR / f"solver_times_{RESOLUTION}_{SAMPLE_INDEX:010d}.csv"
csv_path.parent.mkdir(parents=True, exist_ok=True)

timings_df.to_csv(csv_path, index=False)
print(f"Saved timings to: {csv_path}")

**Note:** The *Grid* solvers (`grid:astar`, `grid:dijkstra`) are internal NumPy‑based implementations that avoid constructing a NetworkX graph. They typically run faster and use less memory for large grids (e.g., 1024, 2048, 4096). Consider using these when performance is a priority.

## Results Summary

Sort the solvers by execution time.

In [ ]:
df = timings_df.sort_values("Time (ms)").reset_index(drop=True)
df

## Timing Bar Chart

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(df["Solver"], df["Time (ms)"], color="steelblue")
ax.set_xlabel("Time (ms)")
ax.set_title(f"Solver Comparison - {RESOLUTION}x{RESOLUTION}")
ax.invert_yaxis()

for bar, t in zip(bars, df["Time (ms)"]):
    ax.text(
        bar.get_width() + max(df["Time (ms)"]) * 0.01,
        bar.get_y() + bar.get_height() / 2,
        f"{t:.1f} ms",
        va="center",
        fontsize=9,
    )

plt.tight_layout()
plt.show()

## Path Visualization (All Solvers)

Overlay all solver paths on the elevation map.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(elevation_norm, cmap="terrain", alpha=0.85)

colors = plt.cm.tab20(np.linspace(0, 1, len(results)))

for color, result in zip(colors, results):
    path = result["path"]
    if len(path) == 0:
        continue
    path_arr = np.array(path)
    ax.plot(path_arr[:, 1], path_arr[:, 0], color=color, linewidth=1.2, alpha=0.9, label=result["solver"])

ax.scatter([start[1]], [start[0]], c="lime", s=60, marker="o", label="Start", zorder=5)
ax.scatter([end[1]], [end[0]], c="cyan", s=60, marker="x", label="End", zorder=5)

ax.set_title("All Solver Paths")
ax.axis("off")
ax.legend(loc="center left", bbox_to_anchor=(1, 0.5), fontsize=8)
plt.tight_layout()
plt.show()

## Path Visualization (Per Solver)

Show one subplot per solver to compare route shapes.

In [ ]:
n_solvers = len(results)
cols = min(3, n_solvers)
rows = (n_solvers + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 5 * rows))
if n_solvers == 1:
    axes = np.array([axes])
axes = axes.flatten()

for ax, result in zip(axes, results):
    ax.imshow(elevation_norm, cmap="terrain", alpha=0.85)
    path = result["path"]
    if len(path) > 0:
        path_arr = np.array(path)
        ax.plot(path_arr[:, 1], path_arr[:, 0], "r-", linewidth=1.1, alpha=0.9)
    ax.scatter([start[1]], [start[0]], c="lime", s=40, marker="o", zorder=5)
    ax.scatter([end[1]], [end[0]], c="cyan", s=40, marker="x", zorder=5)
    ax.set_title(f"{result['solver']}\n{result['time_ms']:.1f}ms, {result['path_length']} px")
    ax.axis("off")

for ax in axes[n_solvers:]:
    ax.set_visible(False)

plt.tight_layout()
plt.show()